# 🧠 MACE NLU Fine-Tuning

Fine-tune **Qwen2.5-3B** on MACE NLU training data using **Unsloth + QLoRA**.

**Requirements:**
- Google Colab with **T4 GPU** (free tier works)
- `instruction_data.jsonl` from MACE project

**Output:** `.gguf` file for local Ollama deployment

---

## Cell 1: Install Dependencies
⚠️ If Colab asks to **restart runtime**, do it, then **skip this cell** and continue from Cell 2.

In [ ]:
%%capture
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers
print("✅ Installation complete!")

## Cell 2: Load Base Model (4-bit Quantized)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
print(f"✅ Model loaded! GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## Cell 3: Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
model.print_trainable_parameters()

## Cell 4: Upload Training Data
Upload `instruction_data.jsonl` from `data/nlu/` in your MACE project.

In [ ]:
from google.colab import files

print("📤 Upload instruction_data.jsonl:")
uploaded = files.upload()
TRAINING_FILE = list(uploaded.keys())[0]
print(f"✅ Uploaded: {TRAINING_FILE}")

## Cell 5: Prepare Dataset

In [ ]:
import json
from datasets import Dataset

# Load JSONL
data = []
with open(TRAINING_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue

print(f"📊 Loaded {len(data)} examples")

# Format into ChatML
def format_to_text(example):
    if "messages" in example:
        msgs = example["messages"]
        parts = []
        for m in msgs:
            parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
        return {"text": "\n".join(parts)}
    else:
        sys_msg = example.get("system", "You are MACE-NLU.")
        inp = example.get("input", "")
        out = example.get("output", "")
        return {"text": (
            f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
            f"<|im_start|>user\n{inp}<|im_end|>\n"
            f"<|im_start|>assistant\n{out}<|im_end|>"
        )}

dataset = Dataset.from_list([format_to_text(ex) for ex in data])
print(f"✅ Dataset ready: {len(dataset)} examples")
print(f"\n--- Sample ---")
print(dataset[0]["text"][:400])

## Cell 6: Train 🚀
This takes ~30-60 min on T4 GPU.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("🚀 Training started...")
stats = trainer.train()

print(f"\n{'='*50}")
print(f"✅ TRAINING COMPLETE!")
print(f"{'='*50}")
print(f"Loss:  {stats.metrics['train_loss']:.4f}")
print(f"Time:  {stats.metrics['train_runtime']:.0f}s ({stats.metrics['train_runtime']/60:.1f} min)")
print(f"GPU:   {torch.cuda.max_memory_allocated()/1024**3:.1f} GB peak")

## Cell 7: Test the Fine-Tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

test_inputs = [
    "my name is bob",
    "yo remind me 2 get milk tmrw",
    "remind me to call mom when i get home",
    "no wait, my email is bob@gmail.com not the other one",
    "hey whats up",
    "what is 5 + 3",
    "put that in the other folder",
]

for t in test_inputs:
    inputs = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "You are MACE-NLU. Parse user input into MemoryAction JSON."},
            {"role": "user", "content": f"Parse: {t}"},
        ],
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    out = model.generate(
        input_ids = inputs,
        max_new_tokens = 512,
        temperature = 0.1,
        do_sample = True,
    )
    resp = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n📝 Input:  {t}")
    print(f"📤 Output: {resp[:300]}")
    print("-" * 60)

## Cell 8: Export to GGUF (for Ollama)

In [ ]:
model.save_pretrained_gguf(
    "mace-nlu",
    tokenizer,
    quantization_method = "q4_k_m",
)
print("✅ GGUF exported!")

## Cell 9: Download the GGUF File

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("**/*.gguf", recursive=True)
print(f"📁 Found: {gguf_files}")

if gguf_files:
    print(f"⬇️ Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("❌ No GGUF file found. Check Cell 8 output.")

## 📋 Deployment Instructions

After downloading the `.gguf` file:

### 1. Create `Modelfile`
```
FROM ./mace-nlu-unsloth.Q4_K_M.gguf

TEMPLATE """<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""

SYSTEM "You are MACE-NLU. Parse user input into MemoryAction JSON."
PARAMETER temperature 0.1
PARAMETER stop "<|im_end|>"
```

### 2. Create & Run
```bash
ollama create mace-nlu -f Modelfile
ollama run mace-nlu "Parse: my name is bob"
```